# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** Deborah Maame Araba Koufie
**Student ID:** 46332028

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [24]:
# API-key setup — DO NOT hard-code your key in this cell.

import os
from dotenv import load_dotenv

# Load the API key from the local .env file
load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# OpenAI-compatible client for Groq
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [25]:
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content, response.usage


# Call the function once
answer, usage = ask_llm(
    "In one sentence, explain what artificial intelligence is."
)

print("Answer:")
print(answer)

print("\nToken usage:")
print(usage)

Answer:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and perception, by using algorithms and data to mimic human thought processes.

Token usage:
CompletionUsage(completion_tokens=48, prompt_tokens=51, total_tokens=99, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.040776056, prompt_time=0.005198453, completion_time=0.099996956, total_time=0.105195409)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:** The system gives instructions of how the AI should behave and the user role gives the actual tasks or questions. 

A token is the atomic unit a model reads.

Because longer inputs and outputs require more computation.An exampe is a request containing 2000 tokens costs more resources to process than the one containing 100 tokens.

### Part 1.2 — Temperature: the randomness dial

In [26]:
question = "Suggest a name for a savings product for market traders in Accra."

print("TEMPERATURE = 0.0")
for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=0.0
    )
    print(f"{i+1}. {answer}")

print("\nTEMPERATURE = 1.2")
for i in range(5):
    answer, usage = ask_llm(
        question,
        temperature=1.2
    )
    print(f"{i+1}. {answer}")

TEMPERATURE = 0.0
1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **MarketMate Savings**: This name positions the savings product as a trusted companion for market traders.
5. **Kokroko Savings Plan**: "Kokroko" is a Ghanaian term for a collective savings scheme, which could appeal to market traders who are familiar with this concept.
6. **Accra Trader's Fund**: This name emphasizes the product's focus on supporting market traders in Accra.
7. **Suzyo Savings**: "Suzyo" is a Ghanaian term for "save" or "keep", which could make the product more relatable to market traders.

Choose the one that resonates the most with your target audi

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** The observation i made was that at each temparature there is a variation of the kind of answers the AI produces .AN example in my experiement was that at temparature 0.0 the answers were identical however at temparature 1.2 the answers were more varied and creative allowing different product names to appear in each run.

This showed that increasing the temparature increases the randomness and variety of the responses the model gives.

For the decision-support system i am about to build i will use temparature of 0.0 because since financial decisons require consistency and reliability rather than creativity .I cannot allow the same or similar output to lead to different outputs else it will cause conflicts.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [27]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [28]:
SUMMARY_PROMPT_V1 = """Summarize this:

{letter_text}
"""

for letter_id in ["L002", "L006"]:
    prompt = SUMMARY_PROMPT_V1.format(
        letter_text=LETTERS[letter_id]
    )

    answer, usage = ask_llm(prompt)

    print(f"\n--- {letter_id} ---")
    print(answer)


--- L002 ---
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in his business, but expects it to improve after the festive season and is willing to repay the loan when he can. He doesn't have collateral to offer at the moment.

--- L006 ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and is willing to rely on his personal trustworthiness as collateral.


In [29]:
SUMMARY_SYSTEM_V2 = """
You are an assistant to a microfinance loan officer.

Summarize loan applications in a factual and neutral way.

Rules:
- Use only information provided in the application.
- Do not invent or assume missing details.
- Include the loan amount, purpose, repayment information, and important financial or risk information when available.
- Write only 3 to 4 sentences.
"""

SUMMARY_USER_V2 = """Summarize this loan application:

{letter_text}
"""

for letter_id in ["L002", "L006"]:
    user_prompt = SUMMARY_USER_V2.format(
        letter_text=LETTERS[letter_id]
    )

    answer, usage = ask_llm(
        user_prompt,
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0
    )

    print(f"\n--- {letter_id} V2 ---")
    print(answer)


--- L002 V2 ---
Kwame Boateng, a commercial driver, has applied for a loan of GHS 25,000 to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season. The applicant does not have collateral to secure the loan and has not specified a repayment schedule, stating only that he can pay back when he has the money. The loan is needed urgently, according to the applicant.

--- L006 V2 ---
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business. The loan will be repaid in one year, according to the applicant's plan. There is no collateral offered to secure the loan, but the applicant claims to be trustworthy. The applicant is 22 years old with no prior business experience, but reports being considered "business-minded" by friends.


**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
V1 produced reasonable summaries however it was very vague with the summaries because we onlytold the model to summarize it without stating the specific formats that we wanted the summary in and also stating the details we want to be represen ted in the summary.This means that we cannot guarantee that the model will automatically focus on the information important to a loan officer or it will avoid making assumptions.An example is that in V1 it sumarized L002 as kwame Boateng seeking GHS 25,000 to repair his vehicle and pay personal debt and follows instructions to focus on financial and risk-related details.



In V2 the summary directly states that he has applied for  a loan of GHC25000to use to repair his trotro vehicle and also use it to pay his persoanl debts This shows that V2 is more detailed and points the imporatance of the loan to the loan officer and gives him reasons to give the loan out instead of a summary of why the loan was needed.

*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*
This instruction is important because the AI should only use the information the applicant actually provided. If it makes up details such as income or collateral, the loan officer could make a wrong decision based on false information. This is called hallucination, which is when an LLM generates information that was not actually given.



### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [30]:
import json
import pandas as pd

EXTRACT_PROMPT = """
You are extracting structured information from a microfinance loan application.

Return ONLY a valid JSON object with EXACTLY these keys:

{{
    "applicant_name": string or null,
    "amount_ghs": number or null,
    "purpose": string or null,
    "monthly_profit_ghs": number or null,
    "has_collateral_or_guarantor": boolean or null,
    "repayment_months": number or null
}}

Rules:
- Use only information stated in the letter.
- If a field is not stated in the letter, use null.
- Do not guess or invent information.
- Do not add any extra keys.
- Return only JSON, with no explanation.

Example:

Letter:
"My name is Ama Kusi. I am requesting GHS 5,000 to buy baking equipment
for my bakery. My monthly profit is GHS 700. My father will guarantee
the loan. I plan to repay over 10 months."

Output:
{{
    "applicant_name": "Ama Kusi",
    "amount_ghs": 5000,
    "purpose": "buy baking equipment for bakery",
    "monthly_profit_ghs": 700,
    "has_collateral_or_guarantor": true,
    "repayment_months": 10
}}

Now extract the information from this letter:

{letter_text}
"""



def extract_fields(letter_text, temperature=0.0):
    prompt = EXTRACT_PROMPT.format(letter_text=letter_text)

    answer, usage = ask_llm(
    prompt,
    temperature=temperature
)

    # Remove possible Markdown JSON fences
    cleaned = answer.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned[len("```json"):]

    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]

    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]

    cleaned = cleaned.strip()

    # Convert JSON text into a Python dictionary
    try:
        return json.loads(cleaned)

    except json.JSONDecodeError:
        print("Warning: Could not parse the model output as JSON.")
        return None


results = []

for letter_id, letter_text in LETTERS.items():
    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)


df_extracted = pd.DataFrame(results)

df_extracted

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months,letter_id
0,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0,L001
1,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN,L002
2,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0,L003
3,Yaw Owusu,12000,for feed and 500 new layers for poultry farm,1500.0,True,18.0,L004
4,None,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0,L005
5,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0,L006


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
The few shot example must not come from the six letters because when we give the model one of the six letters it will make the evaluation unfair because .Using the made up example makes the model fair because it tells the model the required format instead of showing the test answers.


*2. Why "use null, do not guess" — what did the model do without that instruction?*
The instruction to use null prevents the model from adding information that was not in the loan application.An exmple is L002 does not state a specific monthly profit or repayment method so the model returned missing values instead of creating values.Without this instructio the model might assume or bring up possible values for the financial information which will be misleading and lead to hallucination and will mislead the loan officer.


*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*
Temparature 0 is suitable because we want the model to be consistent and then using the temparature at any value other than 0 like 0.2 wil make the model creative however we do not want the model to be creative and assume rather we want it to be consistent with the data and information it is producing because we want consistency because it is a financial system and we do not want to confuse the loan officer.


### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [31]:
BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer.

Using the original loan application and the extracted JSON, prepare a decision-support brief with exactly these sections:

1. Strengths
- Use bullet points.
- Include only strengths supported by the application.

2. Risks / red flags
- Use bullet points.
- Include only risks supported by the application.

3. Missing information
- State important information the loan officer should request if it is not provided.

4. Suggested next step
- Suggest an appropriate action such as inviting the applicant for an interview,
  requesting supporting documents, or flagging the case for senior review.

Rules:
- Use only information provided in the application and extracted JSON.
- Do not invent or assume missing details.
- Be factual and neutral.
- Do NOT say "approve" or "reject".
- The final lending decision must be made by a human loan officer.

Original loan application:
{letter_text}

Extracted JSON:
{extracted_json}
"""

# Create a dictionary of the extraction results from Section 3.2
extracted_by_id = {
    item["letter_id"]: item
    for item in results
}

briefs = {}

for letter_id, letter_text in LETTERS.items():

    extracted = extracted_by_id[letter_id].copy()

    # Remove letter_id because it is not one of the six extracted fields
    extracted.pop("letter_id", None)

    prompt = BRIEF_PROMPT.format(
        letter_text=letter_text,
        extracted_json=json.dumps(extracted, indent=2)
    )

    answer, usage = ask_llm(
        prompt,
        temperature=0.0
    )

    briefs[letter_id] = answer

for letter_id in ["L001", "L002", "L006"]:
    print(f"\n--- {letter_id} DECISION-SUPPORT BRIEF ---")
    print(briefs[letter_id])


--- L001 DECISION-SUPPORT BRIEF ---
## 1. Strengths
* The applicant, Akosua Mensah, has 12 years of experience selling provisions at Makola Market.
* She has a stable business with a monthly profit of GHS 900.
* Akosua has saved GHS 2,500 with the susu scheme over two years without missing a contribution, demonstrating her ability to manage savings and potentially repay a loan.
* She has a guarantor, her sister, who is a teacher, adding an extra layer of security to the loan.
* The applicant has a clear plan for loan repayment, proposing to pay GHS 450 monthly over 20 months.

## 2. Risks / red flags
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which might pose a risk if the business does not generate enough income to cover the loan repayments.
* Expanding into frozen foods with a deep freezer could introduce new operational risks and costs that might affect the applicant's ability to repay the loan.

## 3. Missing information
Im

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*

The system identified the main strengths and red flags correctly.For L300 it recognized the strenth such as Efua having an established registered  business an average monthly profit of GHS2800 a GHS50000 fixed deposit that can be pledged ,sales record and clear a repayment plan.In contrast L006 H=had major red flags because Kofi has not started any of them three proposed businesses,has no current profits information and collateral,and expects to repya it mainly on  the assumption that the business will succeed .This is evidence that the system will be able to distinguish between a relatively well supported application and one with much uncertainty. 
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

We approve the model of having an "approved" or "decline" decision because logically speaking the AI does not have all teh required information needed to make a final lending decision and a human officer may need to verify documents ,ask further questions and use other relevant information.Relating AI to ethics fully automated decisions will unfairly affect applicants if the AI makes incorrect or biased judgement.Keeping the human loan officer provides a watch allowing the AI the decision rather than make it on its own.



### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** 99cdc63

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [32]:
# Fields we want to evaluate
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months"
]

comparison_rows = []

# Compare L001, L003 and L006
for letter_id, gold_values in GOLD.items():

    # Get our extracted row for this letter
    extracted_row = df_extracted[
        df_extracted["letter_id"] == letter_id
    ].iloc[0]

    for field in fields:
        extracted_value = extracted_row[field]
        gold_value = gold_values[field]

        # Treat NaN from pandas and None from GOLD as the same missing value
        if pd.isna(extracted_value) and gold_value is None:
            correct = True
        else:
            correct = extracted_value == gold_value

        comparison_rows.append({
            "letter_id": letter_id,
            "field": field,
            "extracted": extracted_value,
            "gold": gold_value,
            "correct": correct
        })


comparison_df = pd.DataFrame(comparison_rows)

comparison_df

field_accuracy = (
    comparison_df
    .groupby("field")["correct"]
    .mean()
    .mul(100)
    .reset_index()
)

field_accuracy.columns = ["field", "accuracy_percent"]

field_accuracy

,field,accuracy_percent
0,amount_ghs,100.0
1,applicant_name,100.0
2,has_collateral_or_guarantor,100.0
3,monthly_profit_ghs,100.0
4,purpose,0.0
5,repayment_months,100.0


### Part 4.2 — Reliability: is the system consistent?

In [33]:
results_temp_0 = []
results_temp_1 = []

# Run L004 five times at temperature = 0.0
for i in range(5):
    result = extract_fields(
        LETTERS["L004"],
        temperature=0.0
    )
    results_temp_0.append(result)

# Run L004 five times at temperature = 1.0
for i in range(5):
    result = extract_fields(
        LETTERS["L004"],
        temperature=1.0
    )
    results_temp_1.append(result)


def check_reliability(results, temperature):
    valid_results = [
        result for result in results
        if result is not None
    ]

    valid_count = len(valid_results)

    unique_results = set(
        json.dumps(result, sort_keys=True)
        for result in valid_results
    )

    print(f"Temperature = {temperature}")
    print(f"Valid JSON: {valid_count}/5")
    print(f"Unique outputs: {len(unique_results)}")

    if valid_count == 5 and len(unique_results) == 1:
        print("All 5 runs produced identical values.")
    else:
        print("The values were not identical across all 5 runs.")


check_reliability(results_temp_0, 0.0)

print()

check_reliability(results_temp_1, 1.0)

Temperature = 0.0
Valid JSON: 5/5
Unique outputs: 1
All 5 runs produced identical values.

Temperature = 1.0
Valid JSON: 5/5
Unique outputs: 1
All 5 runs produced identical values.


### Part 4.3 — Hallucination probing

In [34]:
# Test 1: Ask for a detail that is NOT in the letter

test1_prompt = f"""
Here is a loan application:

{LETTERS["L001"]}

What is the applicant's credit score?
"""

test1_answer, usage = ask_llm(
    test1_prompt,
    system_prompt=SUMMARY_SYSTEM_V2,
    temperature=0.0
)

print("--- TEST 1: Missing detail ---")
print(test1_answer)

--- TEST 1: Missing detail ---
The application does not mention the applicant's credit score. The applicant has a history of consistent savings with the susu scheme and proposes a repayment plan, but a credit score is not provided. The loan amount requested is GHS 8,000 to expand the business. The applicant's monthly profit is GHS 900 and proposed monthly repayment is GHS 450.


In [35]:
# Test 2: Feed irrelevant text to the extractor

irrelevant_text = """
The weather in Accra was cloudy this morning with light rainfall.
Temperatures are expected to remain warm throughout the afternoon.
Residents are advised to carry umbrellas.
"""

test2_result = extract_fields(
    irrelevant_text,
    temperature=0.0
)

print("\n--- TEST 2: Irrelevant text ---")
print(test2_result)


--- TEST 2: Irrelevant text ---
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


Test 1 — PASS
Output: The application does not mention the applicant's credit score...
Reason: The model admitted that the requested information was absent instead of inventing a credit score.

Test 2 — PASS
Output: {'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}
Reason: The extractor correctly returned missing values for irrelevant input instead of fabricating loan information.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
The purpose field was hardest for the model because 5 fields were 100% while purpose was 0% under exact string matching.The purpose field was semantically correct but were phrased differently from the gold labels.This shows that exact string matching can underestimate performance for free text fields because two descriptions can hve the same meaning without using identical wording.

*2. What did the reliability experiment show about temperature and production systems?*
The reliability experiment showed that the lower temparature produced more consistent outputs,at temparature 0.0 all the 5 runs produced valid JSON and identical values while at temparature 1.0 all the 5 runs were valid JSON but there were 2 unique outputs.This suggests that production systems should prefer a low temparature for structured decision support tasks where consistency and reliability are more important than creativity.

*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*
The system did not hallucinate during the two probing tests. When asked for a credit score that was not in the application, it correctly stated that the information was not provided. When given irrelevant weather text, the extractor returned None for all fields instead of inventing applicant information. The risk can be further reduced by keeping prompts explicit, instructing the model not to guess, using null for missing information, validating structured outputs, and keeping a human reviewer in the loop.


### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
If the applicants write poorly in English but run solid businesses they could be harmed unfairly.One may have a profitable and reliable business but provide a weak application causing the AI to interpret missing details as a financial weakness.This could make the system judge writing ability instead of the applicant’s actual ability to repay.


*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
Since the loan contains sensitive information sending it to a third party API in another country creates privacy and security concerns.Before deploying the system, I would carefully review the API provider’s terms and conditions and privacy policy to find out how the information is stored, how long it is retained, who can access it, and whether it may be used for model training, research purposes, or shared with researchers or other third parties. I would also check the security measures used to protect the data and make sure applicants are informed about how their information will be processed. This would help protect users’ privacy and reduce the risk of their personal information being misused.

*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

Bias Checks:Regularly test the system to see whether it performs worse for certain groups, writing styles, or types of applicants, and review the model if unfair patterns appear.

human review:A trained loan officer should verify the AI’s output and make the final lending decision instead of allowing the AI to approve or reject applications automatically.



---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
   Iteraiting on prompts is simiar to fine tuning a hyperparameter because i can make a change ,test the results,evaluate the performance and improve it again.However when using hyperparameter it changes how the model learns because it changes the learning rate,network size.However ,prompt engineering does not retrain the LLM instead i improve the instructions i give to an already trained model to allow it to give more precise and accurate answers and make sure it is suitable for the task.


2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   No, I would not trust this system to run unattended because financial decision-support requires consistency and reliability. The evaluation result that influenced me most was the reliability test: at temperature 0.0, all five runs produced identical outputs, but at temperature 1.0, the system produced two different outputs across the five runs. This shows that the model can become less consistent when randomness increases, which is risky in a financial setting. Therefore, I think human oversight is still necessary after the AI has completed its part.

3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
   For one of my response.usage results,a call used 99 tokens .Using this as a rough estimate ,processing 1000 applications will require 99,000 tokens per month.However ,the actual number will be higher because each application in the system requires longer prompts and multiple prompts for summarization,extraction and decsion supoort brief.This means provider choice should consider the cost per token,rate limits ,reliabaility,privacy policies and wheter the provider can handle the expected volume efficiently.

4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?
   Calling a foundation model through an API is better for this task because the model is already trained to understand and generate natural language.,so we do not need a large labelled dataset and expensive computing resources ,or a long training process,.We can also improve its performance through prompt engineering quickly how i did in thei slab.However,using an API may not be the best option when the data is highly sensitive,when an organization needs full control over how the model operates, when the internet access is unavailable or when the task is very specific and there is enough data and resources to train a custom model.



---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.